In [11]:
# ==========================================
# TEST EN DIRECT - RECONNAISSANCE D'ÉMOTIONS
# ==========================================

import sounddevice as sd
from scipy.io.wavfile import write
import librosa
import numpy as np
import tensorflow as tf
import joblib
import time
import noisereduce as nr

# 1. Configuration de l'enregistrement
duree = 5  # Durée en secondes
frequence = 22050  # Fréquence standard (idéale pour librosa)

print(f"🎤 Prépare-toi... Enregistrement de {duree} secondes dans :")

# Un vrai compte à rebours qui s'affiche seconde par seconde
for i in range(3, 0, -1):
    print(f"{i}...", flush=True) 
    time.sleep(1)

print("🔴 PARLE MAINTENANT !", flush=True)

# Démarrer l'enregistrement
audio = sd.rec(int(duree * frequence), samplerate=frequence, channels=1)
sd.wait()  # Attendre la fin stricte des 5 secondes
print("⏹️ Enregistrement terminé. Nettoyage du bruit en cours...", flush=True)

# --- NOUVEAU : Réduction du bruit ---
# On aplatit l'audio pour le filtre, puis on le nettoie
audio_plat = audio.flatten()
audio_propre = nr.reduce_noise(y=audio_plat, sr=frequence)

# Sauvegarder temporairement l'audio nettoyé
fichier_test = "test_micro.wav"
write(fichier_test, frequence, audio_propre)

print("🧠 Analyse par l'Intelligence Artificielle...", flush=True)

# 2. Chargement du modèle et du scaler (le "cerveau" et son ajusteur)
model = tf.keras.models.load_model('emotion_recognition_model.keras')
scaler = joblib.load('scaler.save')

# 3. Extraction des caractéristiques (comme pendant l'entraînement)
y, sr = librosa.load(fichier_test, sr=None)
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
mfccs_mean = np.mean(mfccs.T, axis=0)

# 4. Préparation de la donnée pour le CNN
# On applique la même échelle
mfccs_scaled = scaler.transform([mfccs_mean])
# On redimensionne (1 fichier, 40 variables, 1 dimension)
donnee_prete = np.expand_dims(mfccs_scaled, axis=2)

# 5. La Prédiction
prediction = model.predict(donnee_prete, verbose=0)
index_emotion = np.argmax(prediction)

# Dictionnaire des émotions (basé sur l'ordre RAVDESS)
emotions_dict = {
    0: "Neutre 😐",
    1: "Calme 😌",
    2: "Heureux 😄",
    3: "Triste 😢",
    4: "En colère 😠",
    5: "Effrayé 😨",
    6: "Dégoûté 🤢",
    7: "Surpris 😲"
}

resultat = emotions_dict.get(index_emotion, "Inconnue")
confiance = np.max(prediction) * 100

# Affichage des résultats
print("\n" + "=" * 35)
print(f"🎯 Émotion détectée : {resultat}")
print(f"📊 Indice de confiance : {confiance:.2f}%")
print("=" * 35)

🎤 Prépare-toi... Enregistrement de 5 secondes dans :
3...


c:\Users\HP\CodeAlpha_EmotionRecognition\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2...
1...
🔴 PARLE MAINTENANT !
⏹️ Enregistrement terminé. Nettoyage du bruit en cours...
🧠 Analyse par l'Intelligence Artificielle...

🎯 Émotion détectée : Dégoûté 🤢
📊 Indice de confiance : 98.61%
